# The Birthday Paradox in Truncated SHA-256
## *How many hashes until two collide?*

Imagine standing in a classroom doorway are recording each student's birthday as they arrive to class (excluding leap year birthdays).

$$
\begin{align*}
P(\text{no collisions among 1 person}) &= 1\\
P(\text{no collisions among $2$ people}) &= \left(1-\frac{1}{365}\right) = 0.99726\\
P(\text{no collisions among $3$ people}) &= \left(1-\frac{1}{365}\right)\left(1-\frac{2}{365}\right) = 0.991796\\ 
P(\text{no collisions among $4$ people}) &= \left(1-\frac{1}{365}\right)\left(1-\frac{2}{365}\right)\left(1-\frac{3}{365}\right)=0.983644\\
&\ \ \vdots  \\ %lines up spacing
\\  
\end{align*}
$$
For $k$ independently modeled birthdays uniformly distributed over $365$ days $\text{[Paar } \S \text{11.2.3 p. 342-343]}$:
$$
\begin{align*} \\
P(\text{no collisions among $k$}) &= \prod_{i=1}^{k-1} \left( 1 - \frac{i}{365} \right) \\
P(\text{at least 1 collision}) &= 1 - \prod_{i=1}^{k-1} \left( 1 - \frac{i}{365} \right) 
\end{align*}
$$


For $k = 23$, 
$$
\begin{align*}\\
P(\text{at least 1 collision}) &= 1- \prod_{i=1}^{22} \left( 1 - \frac{i}{365} \right)  \approx0.507297 \qquad 
\end{align*}
$$


>[Paar] C. Paar, J. Pelzl, and T. Güneysu, *Understanding Cryptography: From Established Symmetric and Asymmetric Ciphers to Post-Quantum Algorithms*, 2nd ed. Berlin, Heidelberg: Springer, 2024, doi: 10.1007/978-3-662-69007-9.

Here distinct messages replace human subjects. The first $b$ bits of the SHA-256 hash digests we examine takes the place of birthdays. 

A hash function takes an input message and generates a "label" or "fingerprint" of the message called a digest.

Hash functions are deterministic, meaning the same input message will always produce the same output digest.

Rather than 365 possible birthdays, we have $M=2^b$ possible digests. A repeated hash digest is considered a collision.

$$
P(\text{no collisions among $k$}) = \prod_{i=1}^{k-1} \left( 1 - \frac{i}{M} \right)
$$

Consider how collisions might be used for two messages.

If collisions happen sooner than we may expect, we could modify non-visible characters in the message and hash repeatedly until we find a match. 

This is known as *Yuval's birthday attack*. $\text{[Paar } \S \text{11.2.3 p. 341]}$

$$
\begin{align*}
x_1 &= \texttt{Transfer \$5 from Penny the Penguin to Pete the Penguin} \\
x_2 &= \texttt{Transfer \$50,000 from Penny the Penguin to Pete the Penguin} \\
\end{align*}
$$

For this experiment, we count one number per trial: $T$, the number of messages hashed including the repeated hash prefix. This is the hashes-until-collision value. 

A trial that ends at $T=6$ has given $6$ distinct messages to the hash function and seen only 5 distinct labels, implying the 6th distinct message produced a hash collision. 

Start with the one-byte ($b=8$ bits) example. To generate another dataset, change the three settings under Run the Experiment, then rerun all the cells with a Python kernel that has  $\texttt{NumPy}$ installed.

In [1]:
import csv
import hashlib
from pathlib import Path

import numpy as np

# %load_ext rpy2.ipython

# Making Each Message Unique

A trial has identifier $r$, and its messages are numbered $i=1,2,3,\ldots$. 

Store each number in $8$ bytes, most significant byte first, and join them. 

Use a fresh $r$ for each trial to keep the message sets disjoint. 

In [2]:
U64_MAX = 2**64 - 1  # Maximum value of a 64-bit unsigned integer

In [3]:
from scripts.simulation import integer_in_range


from scripts.simulation import encode


# Turn a Digest into a Label
SHA-256 returns $256$ bits. 

We keep the leading $b$ bits and read them as an integer from $0$ to $2^b-1$. 

$8$ bits ($1$ byte) gives $256$ labels and $16$ bits ($2$ bytes) gives $65,536$. The width is measured in bits $b$. 

Non-byte widths work as well. 

For a $12$-bit label, we read the first $16$ bits and discard the last $4$ bits. 

The digest is read as `eb29` and becomes `eb2`.  

In [4]:
from scripts.simulation import truncate_sha_256


# Following One Trial Through The Algorithm

We start with an empty dictionary that keeps track of draws and their unique labels. 

After we encode a unique message, we apply the hash function and take the prefix of the digest which becomes the label. 

Before we increment the draw counter, we ask if the label has already appeared in the dictionary and if so, return the collision. 

Otherwise record where exactly the unique label appears. 

With $M$ labels, the pigeonhole principle guarantees a repeat by draw $M+1$ when the counter can reach it. 

The $64$-bit counter begins at $1$ and can reach $2^{64}-1$. 

Small widths such as $8$, $12$, and $16$ make the experiment practical to compute. 

In [5]:
from scripts.simulation import collision_search


In [6]:
demo_T = collision_search(trial_id=82, bits=8, trace=True)
print(f"Recorded waiting time: T = {demo_T}")

Draw Prefix Observation
   1     c8 new
   2     15 new
   3     eb new
   4     6a new
   5     eb matches draw 3
Recorded waiting time: T = 5


# Run the Experiment

In [7]:
# Settings
# ========
BITS = 16
TRIALS = 900
MASTER_SEED = 37430101

In [8]:
from scripts.simulation import experiment_driver


In [9]:
rows = experiment_driver(TRIALS, BITS, MASTER_SEED)
waits = np.array([row["T"] for row in rows])

print(f"{len(rows):,} trials | {sum(row['T'] for row in rows):,} hash evaluations")
print(f"Sample mean = {waits.mean():.3f} | Sample median = {np.median(waits):g}")
print("First row:", rows[0])

900 trials | 284,233 hash evaluations
Sample mean = 315.814 | Sample median = 294
First row: {'trial': 1, 'trial_id': '3931139061786703017', 'bits': 16, 'master_seed': '37430101', 'T': 408}


# What does the Birthday Model Predict?

 Let $S(k)=P(T>k)$ define the probability that the first $k$ labels are all unique. 

 Given $k$ labels, the next draw has $M-k$ choices that let the trial survive, and 

$$
S(0) = 1, \qquad S(k + 1) = S(k)\left(1 - \frac{k}{M}\right), \qquad F(k) = P(T \le k) = 1 - S(k).
$$

For 16 bits, the first $k$ with $F(k)\ge 0.5$ (median) is 302. 

The exact model mean, $E[T]=\sum_{k=0}^{M}S(k)$, is about 321.52 hashes. 

The median answers “by when have half the trials stopped?” and the mean answers “how many hashes does a trial use on average?”

In [10]:
from scripts.simulation import collision_probability


In [11]:
k = round((2 * 2**BITS * np.log(2)) ** 0.5 + 0.5)  # Round to nearest integer
model_p = collision_probability(k, 2**BITS)
observed_count = int(np.count_nonzero(waits <= k))

print(
    f"By draw {k}: model {model_p:.2%}, "
    f"observed {observed_count}/{TRIALS} "
    f"= {observed_count / TRIALS:.2%}"
)

By draw 302: model 50.07%, observed 469/900 = 52.11%


# Save the Trial Data for Analysis 
Run this notebook from the repository root. Trial data is saved as CSV files in `build/data/`.

The filename records width, trial count, and seed settings. 

In [12]:
from scripts.simulation import save_trials


In [13]:
csv_path = save_trials(rows)
print(f"Saved {len(rows)} trials to {csv_path}")

Saved 900 trials to build/data/trials_b16_n900_seed37430101.csv


# Compare Widths

Keep the trial count and seed fixed, then add four bits at a time. 

Four additional bits give $2^4=16$ times as many labels. 

The birthday approximation $P(T>k)\approx\exp[-k(k-1)/(2M)]$ suggests a typical wait proportional to $\sqrt M$. 

That predicts roughly four times the wait for each step from 8 to 12 to 16 bits.

In [14]:
# Settings 
# ========
comparison_bits = (8, 12)

In [15]:
print("Bits      Labels    Median T      Mean T    Mean T / sqrt(M)")
for width in sorted({BITS, *comparison_bits}):
    if width == BITS:
        sample = waits
    else:
        other_rows = experiment_driver(TRIALS, width, MASTER_SEED)
        save_trials(other_rows)
        sample = np.array([row["T"] for row in other_rows])
    labels = 2**width
    print(f"{width:>4} {labels:>11,} {np.median(sample):>11g} "
          f"{sample.mean():>11.3f} {sample.mean() / labels**0.5:>19.3f}")

Bits      Labels    Median T      Mean T    Mean T / sqrt(M)
   8         256          19      20.478               1.280
  12       4,096          79      82.259               1.285
  16      65,536         294     315.814               1.234


# Interpreting Results with R

Open an R console from the repository root after running the notebook.

To run the primary analysis in R: 

```R
source("scripts/analysis.R")
result <- analyze_trials(file.path("build/data/trials_b16_n900_seed37430101.csv"))
```
